In [2]:
import os
print("I am in:", os.getcwd())
print("I can see:", os.listdir())

I am in: c:\Users\hp\Downloads\Digital_Lending_Zip_Files_4\wsC
I can see: ['.DS_Store', 'config.py', 'generate', 'out', 'run.py', 'validate', 'wsD_q2_channel_economics.ipynb', '__pycache__']


In [3]:
import os
for path in ["out", "out/data", "out/data/parquet"]:
    exists = os.path.isdir(path)
    print(f"{path}/  exists?", exists)
    if exists:
        print(f"   contains: {os.listdir(path)}")

out/  exists? True
   contains: ['.DS_Store', 'data', 'meta']
out/data/  exists? True
   contains: ['.DS_Store', 'csv', 'parquet']
out/data/parquet/  exists? True
   contains: ['behaviour_monthly.parquet', 'customers.parquet', 'loans.parquet', 'repayments.parquet', '_loans_preStep5_backup.parquet', '_loan_default.parquet', '_loan_status.parquet']


In [4]:
import pandas as pd

cust  = pd.read_csv("out/data/csv/customers.csv")
loans = pd.read_csv("out/data/csv/loans.csv")
vc    = pd.read_csv("out/meta/value_proxy_components.csv")

# Stitch all three into one big table
df = loans.merge(vc, on="loan_id").merge(
    cust[["customer_id", "acquisition_channel", "cac"]],
    on="customer_id"
)

In [5]:
result = df.groupby("acquisition_channel").agg(
    loans          = ("loan_id",       "count"),
    avg_cac        = ("cac",           "mean"),
    default_rate   = ("default_flag",  "mean"),
    avg_profit     = ("value_proxy",   "mean"),
    median_profit  = ("value_proxy",   "median"),
).round(2)

# Flags the gap between mean and median as a % — anything above ~100% is meaningful skew
result["skew_pct"] = (100 * (result["avg_profit"] - result["median_profit"]) / result["median_profit"]).round(0).astype(int)

print(result.sort_values("avg_profit", ascending=False))

                     loans  avg_cac  default_rate  avg_profit  median_profit  \
acquisition_channel                                                            
Referral              7500   306.39          0.07    10086.27         3594.5   
Partner-embedded     12359   613.16          0.08     9958.92         3497.0   
Digital ads          17134  1221.64          0.08     9720.78         2965.0   
Organic               5039   152.24          0.07     9018.43         3591.0   
DSA                   7568  1021.05          0.08     8313.50         3007.5   

                     skew_pct  
acquisition_channel            
Referral                  181  
Partner-embedded          185  
Digital ads               228  
Organic                   151  
DSA                       176  


In [6]:
# First, build the per-loan ingredients as new columns on the table.
# Each line does one thing.

df["margin_before_cac"] = df["nii"] + df["fee"] - df["loss"] - df["servicing"]
df["monthly_margin"]    = df["margin_before_cac"] / df["months_on_book"].clip(lower=1)
df["loan_payback_mo"]   = df["cac"] / df["monthly_margin"].clip(lower=1)

# Now the groupby is dead simple — just average / sum the columns above per channel.

extras = df.groupby("acquisition_channel").agg(
    payback_mo   = ("loan_payback_mo",   "median"),
    total_margin = ("margin_before_cac", "sum"),
    total_cac    = ("cac",               "sum"),
)
extras["clv_cac"] = extras["total_margin"] / extras["total_cac"]

print(result.join(extras[["payback_mo", "clv_cac"]]))

                     loans  avg_cac  default_rate  avg_profit  median_profit  \
acquisition_channel                                                            
DSA                   7568  1021.05          0.08     8313.50         3007.5   
Digital ads          17134  1221.64          0.08     9720.78         2965.0   
Organic               5039   152.24          0.07     9018.43         3591.0   
Partner-embedded     12359   613.16          0.08     9958.92         3497.0   
Referral              7500   306.39          0.07    10086.27         3594.5   

                     skew_pct  payback_mo    clv_cac  
acquisition_channel                                   
DSA                       176    1.801820   8.942741  
Digital ads               228    2.141912   8.767050  
Organic                   151    0.296523  60.048929  
Partner-embedded          185    1.074683  17.044973  
Referral                  181    0.547440  33.727505  
